# Phase 6 — Exploratory Data Analysis (Urban Crime IQ)
Dataset: `data/processed/crimes_features.parquet` (921,013 rows × 41 columns, 2023-01-01 to 2026-09-10).

**Notes:** September 2026 is incomplete and is excluded from monthly trends. Records with `time_unknown == 1` are excluded from hour-of-day charts and reported separately.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
OUT = '../docs/screenshots' if os.path.basename(os.getcwd()) == 'notebooks' else 'docs/screenshots'
os.makedirs(OUT, exist_ok=True)
PATH = '../data/processed/crimes_features.parquet' if os.path.basename(os.getcwd()) == 'notebooks' else 'data/processed/crimes_features.parquet'

df = pd.read_parquet(PATH)
df['date'] = pd.to_datetime(df['date'])

def save(name):
    plt.tight_layout()
    plt.savefig(f'{OUT}/{name}.png', bbox_inches='tight')
    plt.show()

## 1. Dataset overview and data-quality checks

In [ ]:
print('Shape:', df.shape)
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
display(df.dtypes.to_frame('dtype').T)
miss = df.isna().sum()
miss = miss[miss > 0].to_frame('missing')
miss['pct'] = (miss['missing'] / len(df) * 100).round(3)
display(miss)
print('Duplicate rows:', df.duplicated().sum())
print('time_unknown records:', int(df['time_unknown'].sum()), f"({df['time_unknown'].mean()*100:.2f}%)")
print('Records without coordinates:', int((df['has_coordinates'] == 0).sum()))

## 2. Monthly crime trend (September 2026 excluded)

In [ ]:
d = df[df['date'] < '2026-09-01'].copy()
monthly = d.groupby(d['date'].dt.to_period('M')).size()
monthly.index = monthly.index.to_timestamp()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(monthly.index, monthly.values, marker='o', color='#c0392b')
ax.set_title('Monthly Crime Count (Jan 2023 – Aug 2026)')
ax.set_xlabel('Month'); ax.set_ylabel('Number of crimes')
save('01_monthly_trend')
print('Peak month:', monthly.idxmax().strftime('%b %Y'), int(monthly.max()))
print('Lowest month:', monthly.idxmin().strftime('%b %Y'), int(monthly.min()))

## 3. Crimes by hour of day (unknown-time records shown separately)

In [ ]:
known = df[df['time_unknown'] == 0]
hourly = known.groupby('hour').size()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(hourly.index, hourly.values, color='#2c7fb8')
ax.set_xticks(range(24))
ax.set_title(f'Crimes by Hour of Day (excludes {int(df["time_unknown"].sum()):,} unknown-time records)')
ax.set_xlabel('Hour (0–23)'); ax.set_ylabel('Number of crimes')
save('02_crimes_by_hour')
print('Peak hour:', hourly.idxmax(), '| Quietest hour:', hourly.idxmin())
print('Unknown-time records:', int(df['time_unknown'].sum()))

## 4. Crimes by day of week

In [ ]:
dow = df['day_of_week']
print('day_of_week sample values:', sorted(dow.unique())[:8])
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
if dow.dtype == object or str(dow.dtype) == 'category':
    counts = dow.value_counts().reindex(order)
else:
    counts = dow.value_counts().sort_index()   # numeric coding (check 0 = Monday vs Sunday)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(counts.index.astype(str), counts.values, color='#41ab5d')
ax.set_title('Crimes by Day of Week')
ax.set_xlabel('Day'); ax.set_ylabel('Number of crimes')
plt.xticks(rotation=30)
save('03_crimes_by_dow')

## 5. Distribution of crime groups

In [ ]:
cg = df['crime_group'].value_counts()
display(cg.to_frame('count').assign(pct=(cg / len(df) * 100).round(2)))
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=cg.values, y=cg.index, ax=ax, color='#8856a7')
ax.set_title('Crimes by Crime Group'); ax.set_xlabel('Number of crimes'); ax.set_ylabel('')
save('04_crime_groups')

top = df['primary_type'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, ax=ax, color='#e6550d')
ax.set_title('Top 10 Primary Crime Types'); ax.set_xlabel('Number of crimes'); ax.set_ylabel('')
save('05_top_primary_types')

## 6. Correlation analysis
Correlations are computed at the **community-area × day** level (not per record), since socioeconomic columns are constant within an area.

In [ ]:
g = df.groupby(['community_area', 'date']).agg(
    crimes=('is_violent', 'size'),
    violent_share=('is_violent', 'mean'),
    poverty_rate=('poverty_rate', 'first'),
    unemployment=('unemployment', 'first'),
    hardship_index=('hardship_index', 'first'),
    per_capita_income=('per_capita_income', 'first'),
    population=('population', 'first'),
    temperature=('temperature', 'first'),
    rainfall=('rainfall', 'first'),
).reset_index()
corr = g.drop(columns=['community_area', 'date']).corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix (community area × day)')
save('06_correlation')
print(corr['crimes'].sort_values(ascending=False))

## 7. Violent vs non-violent crime

In [ ]:
df['violence_label'] = df['is_violent'].map({0: 'Non-violent', 1: 'Violent', False: 'Non-violent', True: 'Violent'})
print(df['violence_label'].value_counts(normalize=True).mul(100).round(2))

# by hour
v = known.assign(v=known['is_violent'].astype(int)).groupby(['hour', 'v']).size().unstack(fill_value=0)
v.columns = ['Non-violent', 'Violent']
v.plot(figsize=(11, 4.5), marker='o')
plt.title('Violent vs Non-violent Crimes by Hour'); plt.xlabel('Hour'); plt.ylabel('Number of crimes')
plt.xticks(range(24))
save('07_violent_by_hour')

# by season
s = df.groupby(['season', 'violence_label']).size().unstack()
s.plot(kind='bar', figsize=(8, 4.5))
plt.title('Violent vs Non-violent Crimes by Season'); plt.ylabel('Number of crimes'); plt.xticks(rotation=0)
save('08_violent_by_season')

# top 10 community areas by violent share (min 5,000 crimes)
a = df.groupby('area_name').agg(n=('is_violent', 'size'), share=('is_violent', 'mean'))
a = a[a['n'] >= 5000].sort_values('share', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=a['share'] * 100, y=a.index, ax=ax, color='#b30000')
ax.set_title('Top 10 Areas by Violent-Crime Share (areas with ≥ 5,000 crimes)')
ax.set_xlabel('% violent'); ax.set_ylabel('')
save('09_violent_share_by_area')

## 8. Findings, limitations and conclusions
_Fill in after reviewing the charts above._

**Findings**
- Monthly trend: …
- Peak hours: …
- Day-of-week pattern: …
- Dominant crime groups: …
- Strongest correlations: …
- Violent vs non-violent: …

**Limitations**
- September 2026 is partial and excluded from monthly trends.
- `time_unknown` records are excluded from hourly analysis.
- ~6,314 records have no coordinates; ~4,702 lack a location description.
- Correlation does not imply causation.

**Conclusion**
- …